# 03 · Crowd → Image  (Stage 3 — the RQ1 aggregation study)

**Crowd-Driven Visual Generation.** The payoff: turn a **crowd of words/emojis** into a single coherent image, and compare **aggregation strategies** — the project's core research question (RQ1).

```
crowd → CLIP text embeddings → AGGREGATE (mean / centroid) → one vector
      → guided DDIM sampling (frozen diffusion) → latent → VAE decode → image
```

Inference only — no training (for the non-learnable aggregators). Reuses the frozen Stage-1 VAE and trained Stage-2 diffusion model.

> Runtime → **GPU (T4)**. Needs `vae.pt` + `diffusion.pt` (pulled from Drive or HF).

## 1. Clone the repo & enter the project

In [ ]:
!git clone --branch feature/crowd-driven-visual-generation https://github.com/nishant-kumar109/gen-ai-IISc.git
%cd gen-ai-IISc/projects/crowd-driven-visual-generation

In [ ]:
!pip install -q open-clip-torch matplotlib   # CLIP text encoder + labeled figures
import torch; print('cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Credentials & locate the checkpoints
Mounts Drive and finds `vae.pt` + `diffusion.pt` (Drive first, else pulled from HF — a **read** token is enough here).

In [ ]:
import os, getpass
from google.colab import drive; drive.mount('/content/drive')

hf_token = getpass.getpass("HF token (Enter to skip if checkpoints are on Drive): ").strip()
if hf_token:
    from huggingface_hub import login; login(token=hf_token)

def locate(drive_path, repo, fname):
    if os.path.exists(drive_path):
        return drive_path
    from huggingface_hub import hf_hub_download, whoami
    return hf_hub_download(f"{whoami()['name']}/{repo}", fname)

VAE_CKPT  = locate('/content/drive/MyDrive/crowdgen/vae/vae.pt', 'crowdgen-vae', 'vae.pt')
DIFF_CKPT = locate('/content/drive/MyDrive/crowdgen/diffusion/diffusion.pt', 'crowdgen-diffusion', 'diffusion.pt')
OUT = '/content/drive/MyDrive/crowdgen/figures'; os.makedirs(OUT, exist_ok=True)
print('VAE :', VAE_CKPT); print('DIFF:', DIFF_CKPT)

## 3. Themes study — does each crowd theme render?
One column per theme, one row per aggregator. A crowd of ~300 words/emojis for each theme is encoded, aggregated, and sampled. Look for theme-appropriate palette/mood in each cell.

In [ ]:
!python sample_crowd.py --vae {VAE_CKPT} --diffusion {DIFF_CKPT} \
    --mode themes --aggregators mean,centroid --n 300 --diversity 0.2 \
    --guidance 3.0 --steps 50 --out {OUT}/crowd_themes.png
from IPython.display import Image; Image(f'{OUT}/crowd_themes.png')

## 4. Diversity study — the RQ1 punchline
Same theme, increasing crowd **diversity** (fraction of off-theme noise). This is the core comparison: as the crowd gets noisier, **mean-pool** should drift toward mush (it averages *everything*), while **cluster-centroid** should stay coherent (it locks onto the dominant theme and ignores scattered noise).

In [ ]:
!python sample_crowd.py --vae {VAE_CKPT} --diffusion {DIFF_CKPT} \
    --mode diversity --theme paradise --aggregators mean,centroid \
    --diversities 0.1,0.4,0.7 --n 300 --guidance 3.0 --steps 50 \
    --out {OUT}/crowd_diversity.png
from IPython.display import Image; Image(f'{OUT}/crowd_diversity.png')

## 5. (Optional) Guidance sweep
How strongly the crowd condition steers generation. Higher `w` = follows the crowd more tightly (but can over-saturate); lower `w` = more diverse/loose. Handy for picking the sweet spot for the report figures.

In [ ]:
for w in [1.0, 2.0, 3.0, 5.0]:
    !python sample_crowd.py --vae {VAE_CKPT} --diffusion {DIFF_CKPT} \
        --mode themes --aggregators centroid --themes paradise,fire,love,night,hope \
        --n 300 --diversity 0.2 --guidance {w} --steps 50 --out {OUT}/sweep_w{w}.png
from IPython.display import Image, display
for w in [1.0, 2.0, 3.0, 5.0]:
    print(f'guidance w={w}'); display(Image(f'{OUT}/sweep_w{w}.png'))

## Next — evaluation & the report
Figures saved under `MyDrive/crowdgen/figures/`. These are the RQ1 result figures. What follows:
- **Quantify RQ1**: theme-fidelity (CLIP similarity between the generated image and the theme) and coherence, per aggregator × diversity — turning the visual story into numbers.
- **Learnable aggregators**: train DeepSets / attention-pooling and add them to the comparison.
- Optional: a stronger diffusion run for crisper final art.